# NB 25 — Wyoming Agricultural Baseline Pull (Wave 6, W6-A)

This notebook builds a county-level, source-explicit baseline. It does not substitute estimates when a required source cannot be fetched. All blocked pulls are cached as attempt evidence and registered in `MANUAL_FETCH.md`.

In [1]:
from pathlib import Path
import os, subprocess

ROOT = Path.cwd()
if not (ROOT / 'notebooks').exists():
    ROOT = Path.cwd().parent
NB_PATH = ROOT / 'notebooks/25_wy_ag_baseline_pull.ipynb'
branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=ROOT, text=True).strip()
candidates = [p for p in (ROOT / 'notebooks').glob('25*.ipynb') if p != NB_PATH]
print(f'pwd: {ROOT}')
print(f'branch: {branch}')
print(f'NB 25 pre-write availability: {not candidates}; conflicting candidates: {candidates}')
assert branch == 'w5-verify', f'expected w5-verify, got {branch}'
assert not candidates, 'NB 25 is not free for this ticket'

pwd: /Users/dylanhartman/projects/Energy Modeling/w5-verify
branch: w5-verify
NB 25 pre-write availability: True; conflicting candidates: []


In [2]:
import csv, json
from datetime import date

RAW = ROOT / 'data/raw'
PROCESSED = ROOT / 'data/processed'
RAW.mkdir(exist_ok=True)
PROCESSED.mkdir(exist_ok=True)
VINTAGE = str(date.today())

fiscal = json.loads((PROCESSED / 'wy_county_fiscal_baseline.json').read_text())
WY_COUNTIES = {geoid: record['county_name'] for geoid, record in fiscal['counties'].items()}
assert len(WY_COUNTIES) == 23

SOURCES = [
    ('census_ag_2022', 'Census of Agriculture 2022 county tables', 'https://www.nass.usda.gov/AgCensus/', 'land use, operations, producers, cattle, forage, irrigated acres'),
    ('nass_quickstats', 'USDA NASS Quick Stats', 'https://quickstats.nass.usda.gov/api/api_GET/', 'Wyoming beef cows state total and county cattle/forage series'),
    ('blm_ras', 'BLM Rangeland Administration System', 'https://reports.blm.gov/reports/ras', 'allotment identifier, county, authorized AUM, acreage'),
    ('usfs_grazing', 'USFS grazing allotments', 'https://data.fs.usda.gov/geodata/edw/datasets.php', 'allotment identifier, county, authorized AUM, acreage'),
    ('wy_water_rights', 'Wyoming State Engineer water rights', 'https://seo.wyo.gov/divisions/water-rights', 'county diversion acre-feet and consumptive-use acre-feet'),
    ('rap', 'Rangeland Analysis Platform', 'https://rangelands.app/', 'county invasive annual grass cover current value and 10-year trend'),
    ('bea_farm_proprietor_income', 'BEA CAINC4 farm proprietors income', 'https://apps.bea.gov/itable/?ReqID=70&step=1', 'county farm proprietor income'),
    ('wy_dor_ag_valuation', 'Wyoming DOR agricultural valuation', 'https://dptax.wyo.gov/annual-reports', 'productive-value coefficients by acre class'),
]

attempts = []
for source_id, title, url, fields in SOURCES:
    attempts.append({'source_id': source_id, 'url': url, 'fields_needed': fields, 'vintage': VINTAGE,
                     'status': 'sandbox_blocked', 'error': 'DNS resolution unavailable in execution sandbox; no source value used.'})
(RAW / f'wy_ag_pull_attempts_{VINTAGE}.json').write_text(json.dumps(attempts, indent=2) + '\n')
print(f'cached blocked-pull evidence: {len(attempts)} source attempts')

cached blocked-pull evidence: 8 source attempts


In [3]:
def datum(source_id, unit, confidence='low', notes='Blocked in sandbox; see MANUAL_FETCH.md.'):
    return {'value': None, 'unit': unit, 'year': None, 'source': source_id, 'confidence': confidence, 'notes': notes}

def county_record(geoid, county_name):
    return {
        'geoid': geoid, 'county_name': county_name, 'state': 'WY',
        'land_by_use': {
            'county_area_acres': datum('census_ag_2022', 'acres'),
            'cropland_acres': datum('census_ag_2022', 'acres'),
            'irrigated_acres': datum('census_ag_2022', 'acres'),
            'pastureland_acres': datum('census_ag_2022', 'acres'),
            'rangeland_acres': datum('census_ag_2022', 'acres'),
            'woodland_acres': datum('census_ag_2022', 'acres'),
            'land_in_farms_acres': datum('census_ag_2022', 'acres'),
            'other_land_acres': {**datum('census_ag_2022', 'acres'), 'is_residual_pool': True,
                'derivation': 'county_area_acres - enumerated land-use classes; never absorbs reconciliation error.'},
            'reconciliation_error_acres': datum('census_ag_2022', 'acres', notes='Must equal zero before a populated pull is accepted.'),
        },
        'cattle_and_forage': {
            'beef_cows_head': datum('nass_quickstats', 'head'),
            'forage_acres': datum('census_ag_2022', 'acres'),
            'stocking_rate_aum_per_acre': datum('nass_quickstats', 'AUM/acre', 'low', 'Documented local stocking rate required; capacity is not inferred.'),
            'aum_capacity': datum('nass_quickstats', 'AUM/year', 'low', 'Derived only as forage acres × documented stocking rate.'),
        },
        'federal_aums': {'blm_authorized_aum': datum('blm_ras', 'AUM/year'), 'usfs_authorized_aum': datum('usfs_grazing', 'AUM/year'), 'federal_aum_share': datum('blm_ras + usfs_grazing', 'fraction')},
        'water': {'diversion_acre_feet': datum('wy_water_rights', 'acre-feet/year'), 'consumptive_use_acre_feet': datum('wy_water_rights', 'acre-feet/year')},
        'rap_invasive_cover': {'current_cover_pct': datum('rap', 'percent'), 'ten_year_trend_pct_points': datum('rap', 'percentage-points/10-years'), 'geography': 'county aggregate only; no sub-county invasive claim'},
        'ag_economics': {'operations_count': datum('census_ag_2022', 'operations'), 'producers_count': datum('census_ag_2022', 'producers'), 'farm_proprietor_income_usd': datum('bea_farm_proprietor_income', 'USD/year')},
        'dor_ag_productive_value_coefficients_usd_per_acre': {'linked_ledger': 'data/processed/wy_fiscal_coefficients.json', 'irrigated': datum('wy_dor_ag_valuation', 'USD/acre'), 'dryland': datum('wy_dor_ag_valuation', 'USD/acre'), 'grazing_land': datum('wy_dor_ag_valuation', 'USD/acre')},
    }

baseline = {
    'schema_version': 'wy_county_ag_baseline_v1', 'vintage': VINTAGE,
    'status': 'blocked_pending_manual_source_acquisition',
    'consumers': ['AG1 Dispatch — Wave 6 agricultural scenario layer'],
    'contract_notes': ['Water is represented only as separate diversion and consumptive-use fields.', 'Ag economics use only Census of Agriculture operations/producers and BEA farm proprietor income (D4).', 'DOR productive-value fields are coefficients and link to the fiscal ledger; assessed values are not duplicated.'],
    'counties': {geoid: county_record(geoid, name) for geoid, name in sorted(WY_COUNTIES.items())},
    'credibility_gate': {
        'status': 'BLOCKED',
        'rows': [
            {'check': 'Fremont and Park lead irrigated acreage', 'result': 'BLOCKED', 'investigation': 'Requires Census 2022 county irrigated acreage.'},
            {'check': 'Campbell and Carbon are rangeland-dominated with large federal AUM shares', 'result': 'BLOCKED', 'investigation': 'Requires Census land use plus BLM/USFS AUMs.'},
            {'check': 'Teton land-in-farms is a small fraction of county area', 'result': 'BLOCKED', 'investigation': 'Requires Census county area and land-in-farms fields.'},
            {'check': 'Statewide beef cows within 10% of NASS state total', 'result': 'BLOCKED', 'investigation': 'Requires NASS county and state beef-cow series.'},
        ],
    },
}
(PROCESSED / 'wy_county_ag_baseline.json').write_text(json.dumps(baseline, indent=2) + '\n')
print('wrote 23 null-safe county records; no estimates substituted')

wrote 23 null-safe county records; no estimates substituted


In [4]:
with (PROCESSED / 'wy_grazing_allotments.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['allotment_id', 'agency', 'allotment_name', 'county_geoid', 'county_name', 'authorized_aum', 'authorized_use_acres', 'source_url', 'vintage', 'status', 'manual_fetch_id'], lineterminator='\n')
    writer.writeheader()
with (PROCESSED / 'wy_ag_sources.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['source_id', 'title', 'source_url', 'fields_needed', 'vintage', 'fetch_status'], lineterminator='\n')
    writer.writeheader()
    for source_id, title, url, fields in SOURCES:
        writer.writerow({'source_id': source_id, 'title': title, 'source_url': url, 'fields_needed': fields, 'vintage': VINTAGE, 'fetch_status': 'sandbox_blocked'})
print('wrote empty-but-schema-valid allotment detail and 8-source registry')

wrote empty-but-schema-valid allotment detail and 8-source registry


In [5]:
manual = ROOT / 'MANUAL_FETCH.md'
heading = f'## W6-A — Wyoming Agricultural Baseline Pull ({VINTAGE})'
entry = f'''
{heading}

All endpoints below were DNS-blocked in the execution sandbox. No values were invented. Cache downloaded source files under `data/raw/` with the pull date, then replace only the corresponding `null` fields in `data/processed/wy_county_ag_baseline.json`.

'''
for source_id, title, url, fields in SOURCES:
    entry += f'- **{source_id} — {title}:** `{url}`. Fields needed: {fields}.\n'
if heading not in manual.read_text():
    manual.write_text(manual.read_text() + entry)
print('appended 8 exact manual-fetch requirements')

appended 8 exact manual-fetch requirements


In [6]:
# P / 0 / scope-audit / review evidence: executable contract checks.
import subprocess
result = subprocess.run(['python', '-m', 'pytest', 'tests/test_wy_ag_baseline_contract.py', '-q'], cwd=ROOT, text=True, capture_output=True)
print(result.stdout)
assert result.returncode == 0, result.stderr
print('Review gates: D4 absence, water separation, residual handling, low AUM confidence, ledger linkage, consumers, and manifest hashes checked.')

...                                                                      [100%]
3 passed in 0.01s

Review gates: D4 absence, water separation, residual handling, low AUM confidence, ledger linkage, consumers, and manifest hashes checked.


---
## W6-A Resume — Actual Fetch Results (2026-07-17)

The initial Codex run had all 8 endpoints DNS-blocked in the sandboxed subprocess. Claude Code retried from its own network context. Results below.


In [ ]:
# W6-A Resume: summary of actual fetch results and sanity table
# This cell is documentation only; data was populated by Claude Code external to the notebook.
print('W6-A resume fetch results documented above.')


=== NASS COA 2022 BULK PULL — SUCCESS ===
Source: https://www.nass.usda.gov/datasets/qs.census2022.txt.gz (295MB)
Filter: STATE_FIPS_CODE=56, AGG_LEVEL_DESC=COUNTY, SOURCE_DESC=CENSUS
Rows extracted: 9893
Counties populated: 23 / 23

=== ENDPOINT PROBE RESULTS ===
census_ag_2022      : SUCCESS — bulk download path
nass_quickstats     : BLOCKED — api_key_required (HTTP 401)
blm_ras             : BLOCKED — no_programmatic_api (web portal only; gis.blm.gov 404)
usfs_grazing        : BLOCKED — download_available (50MB shapefile; manual download req)
wy_water_rights     : BLOCKED — url_404 (seo.wyo.gov/divisions/water-rights)
rap                 : BLOCKED — no_public_api (rangelands.app/api/ 404; api.rangelands.app DNS fail)
bea_farm_proprietor : BLOCKED — api_key_required (BEA API code 1)
wy_dor_ag_valuation : BLOCKED — dns_not_found (dptax.wyo.gov)

=== SANITY TABLE ===
Check 1 — Fremont/Park lead irrigated: INVESTIGATION REQUIRED
  2022 COA ranking: Carbon(157,301) > Sublette(120,242) > 